# 🌲 Tutorial Prático: Pipeline de IA, Visão Computacional e Criptografia Forense
### Sistema SC3 DocAudit — Cadeia de Custódia e Auditoria Documental para a Amazônia

---

**Objetivo deste Notebook:**
Este material foi elaborado com rigor técnico e didático para demonstrar, **passo a passo e de forma reproduzível**, o funcionamento das duas grandes engrenagens do **SC3 DocAudit**:

1. **Camada de Inteligência Artificial & Visão Computacional (PDI):**
   - Pré-processamento e restauração de documentos degradados de fiscalização (CLAHE, Deskewing, Binarização Adaptativa).
   - Extração estruturada de entidades forenses via Regex DFA (CAR, CPF com validação Módulo 11, Multas e Legislação).
   - Calibração de Confiança e Política Anti-Alucinação ($\tau = 0.35$).
   - Validação Cruzada Espaçotemporal (Fórmula de Haversine) e Fonética (Distância de Levenshtein).

2. **Camada de Criptografia Forense & Protocolo SC3:**
   - Função de Hash SHA-256 e conformidade com o Código de Processo Penal (Arts. 158-A a 158-F do CPP).
   - Construção da **Árvore de Merkle** (*Merkle Tree*) unindo peças documentais, fotos de satélite/campo, áudios e coordenadas GPS.
   - Demonstração do **Efeito Avalanche**: como a alteração de um único bit no documento invalida a Raiz Criptográfica.
   - Geração e Verificação de **Prova de Inclusão Criptográfica** (*Merkle Audit Proof*) em $O(\log N)$.


## 0. Configuração do Ambiente e Bibliotecas

Importamos as bibliotecas fundamentais para álgebra, processamento de imagem, criptografia e visualização gráfica.


In [ ]:
import os
import re
import math
import hashlib
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Configuração de estilo dos gráficos
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#1E6B52'
plt.rcParams['axes.linewidth'] = 1.2

print("✅ Ambiente configurado com sucesso! Bibliotecas carregadas:")
print(f" - OpenCV: {cv2.__version__}")
print(f" - NumPy: {np.__version__}")
print(f" - Hashlib: SHA-256 Nativo")


---
## 1. Visão Computacional: Processamento Digital de Imagens (PDI)

Documentos de fiscalização ambiental coletados em campo (Novo Progresso, São Félix do Xingu, Altamira) enfrentam condições severas: umidade, dobras, iluminação desuniforme e baixa resolução de scanners portáteis.

O pipeline de PDI restaura o documento antes de enviá-lo ao OCR neural:
1. **Orientação & Deskewing:** Cálculo de momentos geométricos ($\mu_{20}, \mu_{02}, \mu_{11}$) e rotação afim.
2. **Filtro Bilateral:** Suaviza ruídos mantendo bordas nítidas de caracteres.
3. **CLAHE (Equalização de Contraste Adaptativa):** Distribui luminosidade em blocos $8 \times 8$ com corte de saturação.
4. **Binarização Gaussiana Adaptativa:** Calcula limiares locais $T(x,y)$ eliminando sombras de fundo.


In [ ]:
# 1. Carregamento da Imagem de Teste (compatível com execução a partir da raiz ou da pasta notebooks/)
candidatos_path = [
    "sample_docs/01_termo_embargo_novo_progresso.jpg",
    "../sample_docs/01_termo_embargo_novo_progresso.jpg",
    os.path.join(os.path.dirname(os.path.abspath(__file__ if '__file__' in locals() else '.')), "sample_docs/01_termo_embargo_novo_progresso.jpg")
]

sample_path = next((p for p in candidatos_path if os.path.exists(p)), None)

if sample_path:
    img_bgr = cv2.imread(sample_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    print(f"📄 Imagem real carregada de: {sample_path}")
else:
    # Criação de um documento sintético com ruído e sombra para fins didáticos
    print("ℹ️ Carregando modo sintético demonstrativo...")
    img_gray = np.full((600, 800), 240, dtype=np.uint8)
    cv2.putText(img_gray, "INSTITUTO BRASILEIRO DO MEIO AMBIENTE - IBAMA", (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 30, 2)
    cv2.putText(img_gray, "TERMO DE EMBARGO E INTERDICAO N. 2024-PA-008912", (50, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 50, 2)
    cv2.putText(img_gray, "Infrator: AGROPECUARIA RIO NOVO LTDA", (50, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 60, 1)
    cv2.putText(img_gray, "CPF/CNPJ: 04.123.456/0001-89 | CAR: PA-1505007-0012938475892", (50, 230), cv2.FONT_HERSHEY_SIMPLEX, 0.5, 60, 1)
    cv2.putText(img_gray, "Area Embargada: 420.50 ha | Multa: R$ 2.102.500,00", (50, 270), cv2.FONT_HERSHEY_SIMPLEX, 0.6, 40, 2)
    cv2.putText(img_gray, "Infracao: Art. 70 c/c Art. 72, VII da Lei 9.605/1998", (50, 310), cv2.FONT_HERSHEY_SIMPLEX, 0.55, 50, 1)
    
    # Adicionando ruído e gradiente de sombra
    gradient = np.tile(np.linspace(0, 100, 800), (600, 1)).astype(np.uint8)
    noise = np.random.normal(0, 15, (600, 800)).astype(np.uint8)
    img_gray = cv2.subtract(img_gray, gradient)
    img_gray = cv2.add(img_gray, noise)

print(f"Dimensões do documento: {img_gray.shape[1]}x{img_gray.shape[0]} px")


In [ ]:
# 2. Execução das Etapas de PDI

# Etapa A: Filtro Bilateral (preservação de bordas)
denoised = cv2.bilateralFilter(img_gray, d=9, sigmaColor=75, sigmaSpace=75)

# Etapa B: CLAHE (Contrast Limited Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(denoised)

# Etapa C: Binarização Adaptativa Gaussiana
# T(x, y) = mean(Gaussian(x,y)) - C
binary = cv2.adaptiveThreshold(
    enhanced, 
    255, 
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 
    blockSize=11, 
    C=2
)

# Visualização comparativa lado a lado
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title("1. Original com Ruído & Sombras", fontsize=12, fontweight='bold', color='#1E6B52')
axes[0].axis('off')

axes[1].imshow(enhanced, cmap='gray')
axes[1].set_title("2. Bilateral + CLAHE (Contraste)", fontsize=12, fontweight='bold', color='#1E6B52')
axes[1].axis('off')

axes[2].imshow(binary, cmap='gray')
axes[2].set_title("3. Binarização Adaptativa (Pronto p/ OCR)", fontsize=12, fontweight='bold', color='#1E6B52')
axes[2].axis('off')

plt.tight_layout()
plt.show()


---
## 2. Extração Semântica e Autômatos Finitos (Regex Forense)

Após a vetorização dos caracteres pelo OCR, aplicamos um conjunto de **Autômatos Finitos Determinísticos (DFA)** para extrair entidades críticas com estrita tipagem forense.


In [ ]:
# Texto simulado extraído do OCR binarizado
ocr_extracted_text = """
REPÚBLICA FEDERATIVA DO BRASIL - MINISTÉRIO DO MEIO AMBIENTE
INSTITUTO BRASILEIRO DO MEIO AMBIENTE E DOS RECURSOS NATURAIS RENOVÁVEIS - IBAMA
AUTO DE INFRAÇÃO E TERMO DE EMBARGO N. 2024-PA-008912

AUTUADO: AGROPECUARIA RIO NOVO LTDA
CNPJ/CPF: 04.123.456/0001-89
CADASTRO AMBIENTAL RURAL (CAR): PA-1505007-0012938475892
MUNICÍPIO/UF: Novo Progresso - PA
COORDENADAS GEOGRÁFICAS (SAD69 / SIRGAS2000):
LATITUDE: -7.142500 | LONGITUDE: -55.431200

DESCRIÇÃO DA CONDUTA INFRACIONAL:
Promover desmatamento não autorizado de 420.50 hectares de floresta nativa em área de preservação.
Infração capitulada no Art. 70 c/c Art. 72, incisos II e VII da Lei Federal nº 9.605/1998 e Art. 3º do Decreto 6.514/2008.

VALOR DA MULTA APLICADA: R$ 2.102.500,00 (dois milhões, cento e dois mil e quinhentos reais).
DATA DA AUTUAÇÃO: 14/04/2024
"""


In [ ]:
# 1. Validador Algorítmico de CPF (Cálculo dos Dígitos Verificadores Módulo 11)
def validar_cpf(cpf_str: str) -> bool:
    digits = [int(c) for c in re.sub(r'\D', '', cpf_str)]
    if len(digits) != 11 or len(set(digits)) == 1:
        return False
    # Primeiro dígito
    sum1 = sum(d * w for d, w in zip(digits[:9], range(10, 1, -1)))
    d1 = 11 - (sum1 % 11)
    d1 = 0 if d1 >= 10 else d1
    if digits[9] != d1:
        return False
    # Segundo dígito
    sum2 = sum(d * w for d, w in zip(digits[:10], range(11, 1, -1)))
    d2 = 11 - (sum2 % 11)
    d2 = 0 if d2 >= 10 else d2
    return digits[10] == d2

# 2. Parser Regex de Entidades Estruturadas
def extrair_entidades_forenses(text: str) -> dict:
    car_pattern = r'([A-Z]{2}-\d{7}-[A-F0-9]{13,32})'
    cnpj_cpf_pattern = r'(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}|\d{3}\.\d{3}\.\d{3}-\d{2})'
    multa_pattern = r'R\$\s*([\d\.,]+)'
    artigos_pattern = r'(Art\.?\s*\d+.*?9\.?605/1998)'
    coords_pattern = r'LATITUDE:\s*([-\d\.]+)\s*\|\s*LONGITUDE:\s*([-\d\.]+)'
    
    car = re.search(car_pattern, text)
    doc_fiscal = re.search(cnpj_cpf_pattern, text)
    multa = re.search(multa_pattern, text)
    artigo = re.search(artigos_pattern, text, re.IGNORECASE)
    coords = re.search(coords_pattern, text)
    
    return {
        "car": car.group(1) if car else None,
        "documento_autuado": doc_fiscal.group(1) if doc_fiscal else None,
        "multa_declarada": f"R$ {multa.group(1)}" if multa else None,
        "artigo_lei": artigo.group(1) if artigo else None,
        "lat": float(coords.group(1)) if coords else None,
        "lon": float(coords.group(2)) if coords else None
    }

entidades = extrair_entidades_forenses(ocr_extracted_text)
print("📋 Entidades Forenses Extraídas:")
for k, v in entidades.items():
    print(f"  • {k:20}: {v}")


---
## 3. Calibração de Confiança & Política Anti-Alucinação

Modelos neurais probabilísticos podem sofrer com **alucinação de texto** quando o documento possui ruído severo.
O SC3 DocAudit implementa uma política de calibração estrita com limiar $\tau = 0.35$:
- Se a confiança calibrada do modelo para um campo essencial for inferior a $\tau$, o sistema **rejeita a extração automática** e gera um alerta de inconformidade forense.


In [ ]:
# Simulação de Calibração de Confiança por Campo
campos_auditados = [
    {"campo": "CAR (Cadastro Ambiental)", "confianca_bruta": 0.98, "entropia_token": 0.05},
    {"campo": "CNPJ Autuado", "confianca_bruta": 0.94, "entropia_token": 0.08},
    {"campo": "Valor da Multa", "confianca_bruta": 0.89, "entropia_token": 0.12},
    {"campo": "Coordenadas GPS", "confianca_bruta": 0.92, "entropia_token": 0.09},
    {"campo": "Assinatura do Fiscal", "confianca_bruta": 0.28, "entropia_token": 0.72} # Exemplo degradado
]

TAU_THRESHOLD = 0.35

print("🛡️ Auditoria de Calibração de Confiança:")
print("-" * 65)
for item in campos_auditados:
    # Confiança calibrada penalizada pela entropia de distribuição
    conf_calibrada = item["confianca_bruta"] * (1.0 - item["entropia_token"])
    status = "✅ APROVADO" if conf_calibrada >= TAU_THRESHOLD else "🚨 REJEITADO (Alerta de Alucinação)"
    print(f"{item['campo']:25} | Calibrada: {conf_calibrada:.3f} | {status}")


---
## 4. Validação Cruzada Espaçotemporal (Haversine)

Comparamos em $O(1)$ a distância geodésica entre a coordenada declarada no Auto de Infração e o polígono oficial registrado na base do CAR utilizando a **Fórmula de Haversine**:

$$d = 2 R \arcsin \left( \sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)} \right)$$

Onde $R = 6371.0\text{ km}$ é o raio médio da Terra.


In [ ]:
def distancia_haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0 # Raio da Terra em km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi / 2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c

# Coordenada declarada no Auto de Infração
coord_doc = (entidades["lat"], entidades["lon"])

# Coordenada oficial do Centroide do CAR consultado na base pública
coord_car_base = (-7.145000, -55.433000)

distancia_km = distancia_haversine(coord_doc[0], coord_doc[1], coord_car_base[0], coord_car_base[1])
distancia_metros = distancia_km * 1000

print(f"🛰️ Coordenada Documento:  {coord_doc}")
print(f"🛰️ Coordenada Base CAR:   {coord_car_base}")
print(f"📏 Distância Geodésica:   {distancia_metros:.2f} metros ({distancia_km:.4f} km)")

if distancia_km < 5.0:
    print("✅ Conformidade Geodésica: Ponto dentro da tolerância do imóvel rural!")
else:
    print("⚠️ Alerta de Inconsistência Geográfica!")


---
## 5. Protocolo Criptográfico SC3: Cadeia de Custódia e Árvore de Merkle

O Art. 158-A do Código de Processo Penal define a cadeia de custódia como:
> *"O conjunto de todos os procedimentos utilizados para manter e documentar a história cronológica do vestígio coletado em locais ou em vítimas de crimes..."*

Para garantir que nenhuma prova (documento, foto, áudio, GPS) possa ser forjada ou alterada retroativamente, o SC3 gera uma **Árvore de Merkle com SHA-256**.


In [ ]:
def sha256_hash(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

# Evidências coletadas na Ocorrência Fiscalizatória
evidencias = [
    {"tipo": "Documento Principal (Auto de Infração)", "conteudo": ocr_extracted_text.encode('utf-8')},
    {"tipo": "Foto de Campo (Drone Satélite)", "conteudo": b"EVIDENCIA_FOTOGRAFICA_RAW_DRONE_DJI_RGB_20240414"},
    {"tipo": "Áudio de Depoimento do Gerente", "conteudo": b"AUDIO_DEPOIMENTO_TESTEMUNHA_FLAGRANTE_WAV_PCM_16BIT"},
    {"tipo": "Log de Telemetria GPS Fiscal", "conteudo": b"GPS_FIX_GARMIN_NMEA_LAT_-7.1425_LON_-55.4312_TIMESTAMP_1713110400"}
]

# 1. Hashing das Folhas (Leaves)
leaf_hashes = []
print("🍃 Folhas da Árvore de Merkle (Nível 0):")
for i, ev in enumerate(evidencias):
    h = sha256_hash(ev["conteudo"])
    leaf_hashes.append(h)
    print(f"  [L{i}] {ev['tipo']:36} -> {h}")


In [ ]:
# 2. Construção dos Nós Intermediários e Raiz de Merkle
def construir_arvore_merkle(hashes: list) -> tuple:
    camadas = [hashes]
    current = hashes
    
    while len(current) > 1:
        proxima_camada = []
        for i in range(0, len(current), 2):
            left = current[i]
            right = current[i+1] if i+1 < len(current) else current[i] # Duplica se ímpar
            parent = sha256_hash((left + right).encode('utf-8'))
            proxima_camada.append(parent)
        camadas.append(proxima_camada)
        current = proxima_camada
        
    return current[0], camadas

merkle_root_original, camadas = construir_arvore_merkle(leaf_hashes)

print("🌳 Árvore de Merkle Calculada:")
for nivel, camada in enumerate(camadas):
    print(f" - Nível {nivel} ({len(camada)} nós):")
    for n in camada:
        print(f"     {n}")

print(f"\n🎯 MERKLE ROOT OFICIAL: {merkle_root_original}")


---
## 6. Demonstração Forense do Efeito Avalanche (Avalanche Effect)

E se um agente mal-intencionado alterar **apenas 1 caractere** no documento (por exemplo, mudando o valor da multa de `R$ 2.102.500,00` para `R$ 1.102.500,00`)?

Vejamos como o efeito avalanche da função SHA-256 e da Árvore de Merkle detecta a fraude imediatamente.


In [ ]:
# Simulação de Adulteração Maliciosa de 1 Caractere: '2' -> '1'
texto_adulterado = ocr_extracted_text.replace("R$ 2.102.500,00", "R$ 1.102.500,00")

# Recomputamos os hashes com o documento adulterado
evidencias_adulteradas = [
    {"tipo": "Documento Adulterado", "conteudo": texto_adulterado.encode('utf-8')},
    {"tipo": "Foto de Campo (Drone Satélite)", "conteudo": evidencias[1]["conteudo"]},
    {"tipo": "Áudio de Depoimento do Gerente", "conteudo": evidencias[2]["conteudo"]},
    {"tipo": "Log de Telemetria GPS Fiscal", "conteudo": evidencias[3]["conteudo"]}
]

leaf_hashes_adulteradas = [sha256_hash(ev["conteudo"]) for ev in evidencias_adulteradas]
merkle_root_adulterada, _ = construir_arvore_merkle(leaf_hashes_adulteradas)

print("🚨 AUDITORIA DE INTEGRIDADE CRIPTOGRÁFICA:")
print("=" * 75)
print(f"Hash Folha Original [L0]:   {leaf_hashes[0]}")
print(f"Hash Folha Adulterada [L0]: {leaf_hashes_adulteradas[0]}")
print("-" * 75)
print(f"Merkle Root Legítimo:       {merkle_root_original}")
print(f"Merkle Root Adulterado:     {merkle_root_adulterada}")
print("-" * 75)

# Cálculo da divergência em bits (Distância de Hamming)
bits_orig = bin(int(merkle_root_original, 16))[2:].zfill(256)
bits_adul = bin(int(merkle_root_adulterada, 16))[2:].zfill(256)
diff_bits = sum(b1 != b2 for b1, b2 in zip(bits_orig, bits_adul))
pct_dif = (diff_bits / 256.0) * 100

print(f"💥 Efeito Avalanche: {diff_bits} de 256 bits alterados ({pct_dif:.2f}% de divergência total)!")
print("⚖️ Veredito Forense: FRAUDE DOCUMENTAL DETECTADA COM SUCESSO PELO PROTOCOLO SC3.")


---
## 7. Prova de Inclusão Criptográfica (Merkle Audit Proof)

Como comprovar em juízo que o documento $L_0$ faz parte da ocorrência sem precisar expor o conteúdo sigiloso do áudio $L_2$ ou foto $L_1$?

Com o **Merkle Audit Path**, fornecemos apenas os nós irmãos ($O(\log N)$ hashes) necessários para recompor a Raiz.


In [ ]:
# Prova de Inclusão para L0:
# Para provar L0, precisamos do irmão L1 e do nó pai direito H(L2 + L3)

L0 = leaf_hashes[0]
L1 = leaf_hashes[1]
H_L2_L3 = camadas[1][1]

print("📦 Prova de Auditoria para a Peça L0:")
print(f" - Folha Alvo (L0): {L0}")
print(f" - Prova Criptográfica:")
print(f"     1. Irmão Direito (L1):       {L1}")
print(f"     2. Tio Direito (H(L2 + L3)): {H_L2_L3}")

# Reconstrução da Raiz usando apenas a prova:
pai_esquerdo_calculado = sha256_hash((L0 + L1).encode('utf-8'))
raiz_recalculada = sha256_hash((pai_esquerdo_calculado + H_L2_L3).encode('utf-8'))

print("-" * 65)
print(f"Raiz Reconstituída via Prova: {raiz_recalculada}")
print(f"Raiz Oficial Registrada:      {merkle_root_original}")
assert raiz_recalculada == merkle_root_original, "Erro na prova de inclusão!"
print("✅ SUCESSO: Prova matemática aceita em conformidade com o Art. 158-E do CPP!")


---
## 8. Conclusão e Próximos Passos

Neste tutorial prático, demonstramos:
1. Como o pré-processamento de PDI recupera documentos amazônicos degradados.
2. Como a calibração de confiança previne alucinações neurais em dados sensíveis.
3. Como a validação geodésica de Haversine detecta fraudes de localização.
4. Como o **Protocolo SC3 com SHA-256 e Árvore de Merkle** blinda a cadeia de custódia contra qualquer adulteração física ou digital.

Para ver a aplicação web interativa em execução, acesse:
- 🌐 [Aplicação Web SC3 DocAudit](https://amazoniahackathons3cdocaudit.vercel.app/)
- 📖 [Documentação Técnica Completa](https://amazoniahackathons3cdocaudit.vercel.app/saiba-mais)
